# Relevance Scoring and Rerankers for Trustworthy AI & EU AI Act

This notebook implements an advanced RAG system for a legal tech company querying:
- **EU AI Act PDF** — formal regulatory document
- **Trustworthy AI Podcast transcript** — conversational audio transcript

We demonstrate:
1. Document loading with metadata-rich chunking
2. Baseline vector search (ChromaDB + OpenAI embeddings)
3. LLM-based relevance scoring
4. Cross-encoder reranking (sentence-transformers)
5. Metadata filtering
6. Full RAG pipeline with reranking
7. Performance evaluation (before vs after reranking)

## Step 0: Install Dependencies

In [ ]:
!pip install openai chromadb pypdf sentence-transformers python-dotenv pydub -q

## Step 1: Setup and Data Preparation

In [9]:
import os
import json
import time
import re
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple

import openai
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found. Set it in your .env file.")

client = openai.OpenAI(api_key=OPENAI_API_KEY)
print("✅ OpenAI client initialized")

✅ OpenAI client initialized


### 1a: Load EU AI Act PDF

In [10]:
from pypdf import PdfReader

PDF_PATH = "resources/eu_ai_act.pdf"   # adjust path if needed

def load_pdf_with_metadata(pdf_path: str) -> List[Dict]:
    """Load PDF pages and attach rich metadata to each page."""
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        text = text.strip()
        if not text:
            continue

        # Infer section type from page content heuristics
        section = "recital" if text.startswith("(") else "article" if re.search(r"^Article \d+", text, re.MULTILINE) else "annex" if "ANNEX" in text[:50] else "preamble"

        pages.append({
            "text": text,
            "metadata": {
                "source": "eu_ai_act",
                "source_type": "legal_document",
                "page": i + 1,
                "section": section,
                "title": "EU AI Act (Regulation EU 2024/1689)"
            }
        })
    print(f"📄 Loaded {len(pages)} non-empty pages from EU AI Act PDF")
    return pages

eu_pages = load_pdf_with_metadata(PDF_PATH)

📄 Loaded 144 non-empty pages from EU AI Act PDF


### 1b: Load Podcast Transcript (M4A → transcript text)

We use OpenAI Whisper to transcribe the audio file. If you already have a `.txt` transcript, skip the transcription cell and load the file directly.

In [12]:
AUDIO_PATH = "resources/The_Blueprint_For_Trustworthy_AI.m4a"   # adjust path
TRANSCRIPT_CACHE = "resources/podcast_transcript.txt"

# Audio processing constants
MAX_AUDIO_FILE_SIZE = 25 * 1024 * 1024  # 25 MB (Whisper API limit)
CHUNK_DURATION_MS = 10 * 60 * 1000      # 10 minutes per chunk

from pydub import AudioSegment
import io

def check_audio_file_limits(audio_path: str) -> Dict[str, Any]:
    """
    Check if audio file is within acceptable limits.
    
    Returns:
        Dict with 'valid', 'size_mb', 'needs_chunking', and 'message'
    """
    if not os.path.exists(audio_path):
        return {
            "valid": False,
            "message": f"❌ Audio file not found: {audio_path}"
        }
    
    file_size = os.path.getsize(audio_path)
    size_mb = file_size / (1024 * 1024)
    
    result = {
        "valid": file_size <= MAX_AUDIO_FILE_SIZE,
        "size_mb": round(size_mb, 2),
        "needs_chunking": file_size > MAX_AUDIO_FILE_SIZE,
        "message": f"📊 Audio file size: {size_mb:.2f} MB"
    }
    
    if not result["valid"]:
        result["message"] += f" (exceeds {MAX_AUDIO_FILE_SIZE / (1024 * 1024):.0f} MB limit) — will be chunked"
    
    return result


def chunk_audio_file(audio_path: str, chunk_duration_ms: int = CHUNK_DURATION_MS) -> List[str]:
    """
    Split large audio file into smaller chunks for Whisper transcription.
    
    Args:
        audio_path: Path to audio file
        chunk_duration_ms: Duration of each chunk in milliseconds
    
    Returns:
        List of paths to audio chunks (or original file if no chunking needed)
    """
    file_size = os.path.getsize(audio_path)
    
    if file_size <= MAX_AUDIO_FILE_SIZE:
        print(f"✅ File size OK ({file_size / (1024 * 1024):.2f} MB) — no chunking needed")
        return [audio_path]
    
    print(f"🔪 Chunking audio file ({file_size / (1024 * 1024):.2f} MB into {chunk_duration_ms / 1000 / 60:.0f}-min chunks)...")
    
    # Load audio
    audio = AudioSegment.from_file(audio_path)
    print(f"📼 Loaded audio: {len(audio)} ms ({len(audio) / 1000 / 60:.1f} minutes)")
    
    # Create chunks
    chunks = []
    chunk_dir = os.path.join(os.path.dirname(audio_path), "audio_chunks")
    os.makedirs(chunk_dir, exist_ok=True)
    
    for i, start_ms in enumerate(range(0, len(audio), chunk_duration_ms)):
        end_ms = min(start_ms + chunk_duration_ms, len(audio))
        chunk_audio = audio[start_ms:end_ms]
        
        # Export chunk
        chunk_path = os.path.join(chunk_dir, f"chunk_{i:03d}.m4a")
        chunk_audio.export(chunk_path, format="ipod")
        chunks.append(chunk_path)
        
        chunk_size_mb = os.path.getsize(chunk_path) / (1024 * 1024)
        print(f"  ✅ Chunk {i}: {chunk_size_mb:.2f} MB ({(end_ms - start_ms) / 1000 / 60:.1f} min)")
    
    print(f"📦 Created {len(chunks)} audio chunks")
    return chunks


def transcribe_audio(audio_path: str, cache_path: str) -> str:
    """Transcribe audio via OpenAI Whisper, with file size checking, chunking, and caching."""
    if os.path.exists(cache_path):
        print(f"📼 Loading cached transcript from {cache_path}")
        with open(cache_path, "r", encoding="utf-8") as f:
            return f.read()

    # Check file limits
    limits_check = check_audio_file_limits(audio_path)
    print(limits_check["message"])
    
    if not limits_check["valid"]:
        print("⚠️  File size exceeds limit — will be chunked automatically")
    
    # Chunk audio if needed
    audio_chunks = chunk_audio_file(audio_path)
    
    # Transcribe all chunks
    print("🎙️  Transcribing audio with Whisper...")
    all_transcripts = []
    
    for i, chunk_path in enumerate(audio_chunks):
        print(f"  Transcribing chunk {i + 1}/{len(audio_chunks)}...")
        with open(chunk_path, "rb") as audio_file:
            response = client.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file,
                response_format="text"
            )
        all_transcripts.append(response)
    
    # Combine transcripts
    transcript = " ".join(all_transcripts)
    
    # Save transcript
    with open(cache_path, "w", encoding="utf-8") as f:
        f.write(transcript)
    print(f"✅ Transcript saved to {cache_path} ({len(transcript)} chars)")
    
    # Clean up chunks if created
    if len(audio_chunks) > 1:
        chunk_dir = os.path.join(os.path.dirname(audio_path), "audio_chunks")
        for chunk_path in audio_chunks:
            os.remove(chunk_path)
        os.rmdir(chunk_dir)
        print("🗑️  Cleaned up audio chunks")
    
    return transcript

podcast_transcript = transcribe_audio(AUDIO_PATH, TRANSCRIPT_CACHE)
print(f"Transcript preview: {podcast_transcript[:300]}...")

📊 Audio file size: 28.78 MB (exceeds 25 MB limit) — will be chunked
⚠️  File size exceeds limit — will be chunked automatically
🔪 Chunking audio file (28.78 MB into 10-min chunks)...
📼 Loaded audio: 937529 ms (15.6 minutes)
  ✅ Chunk 0: 9.21 MB (10.0 min)
  ✅ Chunk 1: 5.18 MB (5.6 min)
📦 Created 2 audio chunks
🎙️  Transcribing audio with Whisper...
  Transcribing chunk 1/2...
  Transcribing chunk 2/2...
✅ Transcript saved to resources/podcast_transcript.txt (16454 chars)
🗑️  Cleaned up audio chunks
Transcript preview: So imagine for a second you're driving across, I don't know, a massive suspension bridge. Okay. You don't pull over halfway across, get out and demand to see the blueprints, right? You don't interview the welding crew. No. You just, you trust it. You just drive. You trust the bridge. You trust the e...


### 1c: Chunk Documents with Metadata

In [13]:
def chunk_text(text: str, chunk_size: int = 600, overlap: int = 80) -> List[str]:
    """Split text into overlapping chunks by character count."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


def prepare_eu_chunks(pages: List[Dict]) -> List[Dict]:
    """Chunk EU AI Act pages and carry metadata through."""
    all_chunks = []
    for page in pages:
        text_chunks = chunk_text(page["text"], chunk_size=600, overlap=80)
        for j, chunk in enumerate(text_chunks):
            all_chunks.append({
                "text": chunk,
                "metadata": {
                    **page["metadata"],
                    "chunk_index": j,
                    "chunk_id": f"eu_p{page['metadata']['page']}_c{j}"
                }
            })
    return all_chunks


def prepare_podcast_chunks(transcript: str) -> List[Dict]:
    """Chunk podcast transcript with source metadata."""
    # Split on natural sentence breaks first, then chunk
    text_chunks = chunk_text(transcript, chunk_size=600, overlap=80)
    chunks = []
    for j, chunk in enumerate(text_chunks):
        chunks.append({
            "text": chunk,
            "metadata": {
                "source": "trustworthy_ai_podcast",
                "source_type": "podcast_transcript",
                "chunk_index": j,
                "chunk_id": f"pod_c{j}",
                "title": "The Blueprint for Trustworthy AI (Podcast)"
            }
        })
    return chunks


eu_chunks   = prepare_eu_chunks(eu_pages)
pod_chunks  = prepare_podcast_chunks(podcast_transcript)
all_chunks  = eu_chunks + pod_chunks

print(f"📦 EU AI Act chunks  : {len(eu_chunks)}")
print(f"📦 Podcast chunks    : {len(pod_chunks)}")
print(f"📦 Total chunks      : {len(all_chunks)}")
print("\nSample EU chunk metadata:", eu_chunks[0]["metadata"])
print("Sample Pod chunk metadata:", pod_chunks[0]["metadata"])

📦 EU AI Act chunks  : 1268
📦 Podcast chunks    : 32
📦 Total chunks      : 1300

Sample EU chunk metadata: {'source': 'eu_ai_act', 'source_type': 'legal_document', 'page': 1, 'section': 'preamble', 'title': 'EU AI Act (Regulation EU 2024/1689)', 'chunk_index': 0, 'chunk_id': 'eu_p1_c0'}
Sample Pod chunk metadata: {'source': 'trustworthy_ai_podcast', 'source_type': 'podcast_transcript', 'chunk_index': 0, 'chunk_id': 'pod_c0', 'title': 'The Blueprint for Trustworthy AI (Podcast)'}


## Step 2: Generate Embeddings and Build Vector Store

In [14]:
import chromadb
from chromadb.config import Settings

EMBEDDING_MODEL = "text-embedding-3-small"
COLLECTION_NAME = "trustworthy_ai_rag"

# Persistent local ChromaDB
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Delete collection if re-running to avoid duplicate IDs
try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print("🗑️  Deleted existing collection")
except:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)
print(f"✅ Created ChromaDB collection: {COLLECTION_NAME}")

✅ Created ChromaDB collection: trustworthy_ai_rag


In [15]:
def embed_texts(texts: List[str], batch_size: int = 100) -> List[List[float]]:
    """Embed a list of texts in batches to respect API limits."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
        all_embeddings.extend([e.embedding for e in response.data])
        if i % 500 == 0:
            print(f"  Embedded {i + len(batch)}/{len(texts)} texts...")
    return all_embeddings


def index_chunks(chunks: List[Dict], collection) -> None:
    """Embed chunks and insert into ChromaDB collection."""
    texts     = [c["text"] for c in chunks]
    ids       = [c["metadata"]["chunk_id"] for c in chunks]
    metadatas = [c["metadata"] for c in chunks]

    print(f"⚙️  Embedding {len(texts)} chunks...")
    embeddings = embed_texts(texts)

    # ChromaDB upsert in batches of 500
    BATCH = 500
    for i in range(0, len(texts), BATCH):
        collection.add(
            ids=ids[i:i+BATCH],
            embeddings=embeddings[i:i+BATCH],
            documents=texts[i:i+BATCH],
            metadatas=metadatas[i:i+BATCH]
        )
    print(f"✅ Indexed {len(texts)} chunks into ChromaDB")


index_chunks(all_chunks, collection)

⚙️  Embedding 1300 chunks...
  Embedded 100/1300 texts...
  Embedded 600/1300 texts...
  Embedded 1100/1300 texts...
✅ Indexed 1300 chunks into ChromaDB


### Baseline Retrieval Function

In [16]:
def retrieve_baseline(
    query: str,
    n_results: int = 10,
    source_filter: Optional[str] = None
) -> List[Dict]:
    """
    Basic cosine-similarity retrieval from ChromaDB.

    Args:
        query         : User question
        n_results     : How many chunks to retrieve
        source_filter : Optional — 'eu_ai_act' or 'trustworthy_ai_podcast'
    Returns:
        List of result dicts with text, metadata, and distance.
    """
    query_embedding = client.embeddings.create(
        model=EMBEDDING_MODEL, input=[query]
    ).data[0].embedding

    where_clause = {"source": source_filter} if source_filter else None

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        where=where_clause,
        include=["documents", "metadatas", "distances"]
    )

    hits = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        hits.append({
            "text":              doc,
            "metadata":          meta,
            "similarity_score":  1 - dist   # cosine distance → similarity
        })
    return hits


# Quick smoke test
test_results = retrieve_baseline("What are high-risk AI systems?", n_results=3)
print(f"Baseline retrieval returned {len(test_results)} results")
for r in test_results:
    print(f"  [{r['metadata']['source']}] sim={r['similarity_score']:.3f} — {r['text'][:120]}...")

Baseline retrieval returned 3 results
  [eu_ai_act] sim=0.684 — ANNEX III
High-r isk AI sys tems referred to in Ar ticle 6(2)
High-r isk AI systems pursuant to Ar ticle 6(2) are the AI...
  [eu_ai_act] sim=0.653 — manuf actur ing or personal assistance and care should be 
able to safely operate and perfo r ms their functions in comp...
  [eu_ai_act] sim=0.651 — e result of a previously comp let ed human activity ;
(c) the AI system is intende d to det ect decision-making patt er ...


## Step 3: LLM-Based Relevance Scoring (Advanced)

We send each retrieved chunk to an LLM and ask it to score how relevant the chunk is to the query on a 0–10 scale. We then re-rank by a weighted blend of the embedding similarity and LLM score.

In [17]:
RELEVANCE_SCORING_PROMPT = """\
You are a relevance-scoring assistant for a legal and AI research system.

Given a QUERY and a DOCUMENT CHUNK, rate how relevant the chunk is to answering the query.

Respond ONLY with a JSON object in this exact format:
{{"score": <integer 0-10>, "reason": "<one sentence>"}}

0  = completely irrelevant
5  = partially relevant
10 = perfectly answers the query

QUERY:
{query}

DOCUMENT CHUNK:
{chunk}
"""


def llm_score_chunk(query: str, chunk: str) -> Tuple[float, str]:
    """Ask an LLM to score one chunk's relevance. Returns (score 0-1, reason)."""
    prompt = RELEVANCE_SCORING_PROMPT.format(query=query, chunk=chunk[:800])
    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=120
        )
        content = resp.choices[0].message.content.strip()
        parsed  = json.loads(content)
        return parsed["score"] / 10.0, parsed.get("reason", "")
    except Exception as e:
        print(f"  ⚠️  LLM scoring error: {e}")
        return 0.5, "scoring failed"


def llm_rerank(
    query: str,
    hits: List[Dict],
    alpha: float = 0.4   # weight for LLM score; 1-alpha for similarity
) -> List[Dict]:
    """
    Re-rank retrieved chunks using LLM relevance scores.

    Combined score = alpha * llm_score + (1-alpha) * similarity_score
    """
    print(f"  🧠 LLM scoring {len(hits)} chunks...")
    for hit in hits:
        llm_s, reason = llm_score_chunk(query, hit["text"])
        hit["llm_score"]     = llm_s
        hit["llm_reason"]    = reason
        hit["combined_score"] = alpha * llm_s + (1 - alpha) * hit["similarity_score"]

    return sorted(hits, key=lambda x: x["combined_score"], reverse=True)


# Demo
query_demo = "What obligations do providers of high-risk AI systems have?"
baseline_hits = retrieve_baseline(query_demo, n_results=6)
llm_ranked    = llm_rerank(query_demo, baseline_hits)

print("\n📊 LLM-Reranked Results:")
for i, r in enumerate(llm_ranked[:5], 1):
    print(f"  {i}. combined={r['combined_score']:.3f} | sim={r['similarity_score']:.3f} | llm={r['llm_score']:.1f} | [{r['metadata']['source']}]")
    print(f"     {r['text'][:120]}...")

  🧠 LLM scoring 6 chunks...

📊 LLM-Reranked Results:
  1. combined=0.846 | sim=0.743 | llm=1.0 | [eu_ai_act]
     SECTION 3
Oblig ations of pr o viders and deploye rs of high-r isk AI syste ms and other par ties
Ar ticle 16
Obligation...
  2. combined=0.742 | sim=0.703 | llm=0.8 | [eu_ai_act]
     Ar ticle 25
Responsibilities along the AI value chain
1. Any distr ibut or , imp or te r , deplo y er or other third-par...
  3. combined=0.678 | sim=0.664 | llm=0.7 | [eu_ai_act]
     SYSTEMS
Ar ticle 50
T ransparency obligations f or pro viders and deplo y ers of cer ta in AI sys tems
1. Providers shal...
  4. combined=0.618 | sim=0.696 | llm=0.5 | [eu_ai_act]
     the Uni on. Providers of AI syste ms that are 
not high-r isk should be encouraged to create codes of conduct, including...
  5. combined=0.607 | sim=0.678 | llm=0.5 | [eu_ai_act]
     a general-pur pose AI model with syste mic r isk, providers ma y request 
reassessment at the earliest six months af te ...


## Step 4: Cross-Encoder Reranker (Advanced)

Cross-encoders process the **(query, document)** pair jointly and score relevance directly — this is more accurate than embedding similarity but slower.

In [18]:
from sentence_transformers import CrossEncoder

# Load a lightweight cross-encoder model (downloads on first run ~100MB)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✅ Cross-encoder model loaded")

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9523.03it/s]


✅ Cross-encoder model loaded


In [19]:
import numpy as np

def cross_encoder_rerank(query: str, hits: List[Dict]) -> List[Dict]:
    """
    Rerank hits using a cross-encoder model.
    Adds 'ce_score' (raw logit) and 'ce_score_norm' (0-1 sigmoid) to each hit.
    """
    pairs  = [(query, hit["text"]) for hit in hits]
    scores = cross_encoder.predict(pairs)   # raw logits

    # Normalise with sigmoid so scores are comparable across queries
    norm_scores = 1 / (1 + np.exp(-scores))

    for hit, raw, norm in zip(hits, scores, norm_scores):
        hit["ce_score"]      = float(raw)
        hit["ce_score_norm"] = float(norm)

    return sorted(hits, key=lambda x: x["ce_score"], reverse=True)


# Demo
baseline_hits_fresh = retrieve_baseline(query_demo, n_results=10)
ce_ranked = cross_encoder_rerank(query_demo, baseline_hits_fresh)

print("\n📊 Cross-Encoder Reranked Results:")
for i, r in enumerate(ce_ranked[:5], 1):
    print(f"  {i}. ce_score={r['ce_score']:.3f} | sim={r['similarity_score']:.3f} | [{r['metadata']['source']}]")
    print(f"     {r['text'][:120]}...")


📊 Cross-Encoder Reranked Results:
  1. ce_score=4.194 | sim=0.703 | [eu_ai_act]
     Ar ticle 25
Responsibilities along the AI value chain
1. Any distr ibut or , imp or te r , deplo y er or other third-par...
  2. ce_score=3.989 | sim=0.743 | [eu_ai_act]
     SECTION 3
Oblig ations of pr o viders and deploye rs of high-r isk AI syste ms and other par ties
Ar ticle 16
Obligation...
  3. ce_score=3.892 | sim=0.649 | [eu_ai_act]
     to their par ticular nature and in order to ensure a f air shar ing of responsibilities along the 
AI value ch ain, the ...
  4. ce_score=3.884 | sim=0.664 | [eu_ai_act]
     SYSTEMS
Ar ticle 50
T ransparency obligations f or pro viders and deplo y ers of cer ta in AI sys tems
1. Providers shal...
  5. ce_score=2.198 | sim=0.696 | [eu_ai_act]
     the Uni on. Providers of AI syste ms that are 
not high-r isk should be encouraged to create codes of conduct, including...


## Step 5: Metadata Filtering

Filter retrieval to a specific source (legal doc vs podcast) before reranking.

In [20]:
def retrieve_with_filter(
    query: str,
    source: Optional[str] = None,
    section: Optional[str] = None,
    n_results: int = 10,
    reranker: str = "cross_encoder"   # 'none' | 'llm' | 'cross_encoder'
) -> List[Dict]:
    """
    Retrieve with optional metadata filtering and optional reranking.

    Args:
        query      : User query
        source     : Filter by source ('eu_ai_act' | 'trustworthy_ai_podcast')
        section    : Filter by section type ('article' | 'recital' | 'annex')
        n_results  : Number of results to retrieve before reranking
        reranker   : Which reranker to apply
    """
    # Build ChromaDB where clause
    where_clause = None
    if source and section:
        where_clause = {"$and": [{"source": source}, {"section": section}]}
    elif source:
        where_clause = {"source": source}
    elif section:
        where_clause = {"section": section}

    hits = retrieve_baseline(query, n_results=n_results, source_filter=source)

    if reranker == "cross_encoder":
        hits = cross_encoder_rerank(query, hits)
    elif reranker == "llm":
        hits = llm_rerank(query, hits)

    return hits


# Example: only EU AI Act articles
filtered = retrieve_with_filter(
    "What are prohibited AI practices?",
    source="eu_ai_act",
    n_results=5,
    reranker="cross_encoder"
)
print("Filtered + Cross-Encoder results (EU AI Act only):")
for i, r in enumerate(filtered, 1):
    print(f"  {i}. [{r['metadata']['source']} | p{r['metadata'].get('page','?')}] ce={r.get('ce_score', 'N/A'):.3f}")
    print(f"     {r['text'][:150]}...")

Filtered + Cross-Encoder results (EU AI Act only):
  1. [eu_ai_act | p51] ce=8.605
     Ar ticle 5
Prohibited AI practices
1. The f ollowi ng AI practices shall be prohibite d:
(a) the placing on the market, the putting into ser vice or t...
  2. [eu_ai_act | p9] ce=8.015
     tative AI-enabled practices. The prohibitions f or suc h AI practices are complement ar y to the 
pro visions contained in Directive 2005/29/EC of the...
  3. [eu_ai_act | p51] ce=5.759
     Ar ticle 4
AI literacy
Provi ders and deplo y ers of AI syste ms shall take measures to ensure, to their best exte nt, a suffi cient level of AI lite ...
  4. [eu_ai_act | p12] ce=5.291
     to detect the emotional state of individuals in situations related to the w orkplace and 
education should be prohibite d. That prohibition should not...
  5. [eu_ai_act | p9] ce=2.088
     inte ntion to distor t behavio ur where the distor tion results from f actors exter nal to the AI syste m which are outside 
the control of the provid

## Step 6: Full RAG Pipeline with Reranking

In [21]:
SYSTEM_PROMPT = """\
You are an expert AI regulation and trustworthy AI assistant.
Answer the user's question using ONLY the context provided.
If the context is insufficient, say so clearly.
Cite which source (EU AI Act page / Podcast) your answer comes from.
"""


def rag_pipeline(
    query: str,
    n_retrieve: int = 10,
    top_k: int = 5,
    source_filter: Optional[str] = None,
    reranker: str = "cross_encoder"   # 'none' | 'llm' | 'cross_encoder'
) -> Dict:
    """
    Complete RAG pipeline:
      1. Retrieve n_retrieve chunks via embedding similarity
      2. Rerank with specified reranker
      3. Use top_k chunks as context for LLM answer

    Returns dict with answer, sources, and retrieval metadata.
    """
    # 1. Retrieve
    hits = retrieve_baseline(query, n_results=n_retrieve, source_filter=source_filter)

    # 2. Rerank
    if reranker == "cross_encoder":
        hits = cross_encoder_rerank(query, hits)
    elif reranker == "llm":
        hits = llm_rerank(query, hits)
    # else: no reranking — use embedding similarity order

    context_chunks = hits[:top_k]

    # 3. Format context
    context_str = "\n\n---\n\n".join(
        f"[Source: {c['metadata']['source']} | "
        f"Page: {c['metadata'].get('page', 'N/A')}]\n{c['text']}"
        for c in context_chunks
    )

    user_message = f"Context:\n{context_str}\n\nQuestion: {query}"

    # 4. Generate answer
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message}
        ],
        temperature=0.2,
        max_tokens=600
    )

    answer = response.choices[0].message.content

    return {
        "query":    query,
        "answer":   answer,
        "reranker": reranker,
        "sources":  [{"text": c["text"][:200], "metadata": c["metadata"]} for c in context_chunks]
    }


# Demo query
result = rag_pipeline(
    "What are the key obligations for high-risk AI system providers under the EU AI Act?",
    n_retrieve=10,
    top_k=5,
    reranker="cross_encoder"
)

print("=" * 70)
print(f"Q: {result['query']}")
print(f"Reranker: {result['reranker']}")
print("=" * 70)
print(result["answer"])
print("\n📚 Sources used:")
for s in result["sources"]:
    print(f"  - {s['metadata']['source']} | page {s['metadata'].get('page', 'N/A')}")

Q: What are the key obligations for high-risk AI system providers under the EU AI Act?
Reranker: cross_encoder
The key obligations for providers of high-risk AI systems under the EU AI Act include:

1. Ensuring compliance with the requirements set out in Section 2 of the Act (Article 16(a)).
2. Indicating their name, registered trade name or trademark, and contact address on the high-risk AI system or its packaging/documentation (Article 16(b)).
3. Having a quality management system in place (Article 16(c)).

These obligations are designed to ensure accountability and transparency in the deployment of high-risk AI systems. 

(Source: eu_ai_act | Page: 62)

📚 Sources used:
  - eu_ai_act | page 67
  - eu_ai_act | page 24
  - eu_ai_act | page 82
  - eu_ai_act | page 62
  - eu_ai_act | page 54


## Step 7: Evaluate Performance — Before vs After Reranking

We test a set of benchmark queries and manually evaluate whether the **top-1 retrieved chunk** is correct. We compare three conditions:
- **No reranking** (embedding similarity only)
- **LLM reranking**
- **Cross-encoder reranking**

In [23]:
EVAL_QUERIES = [
    {
        "query": "What are prohibited AI practices under the EU AI Act?",
        "expected_source": "eu_ai_act",
        "expected_keywords": ["prohibited", "manipulate", "subliminal", "social scoring"]
    },
    {
        "query": "What does trustworthy AI mean according to the podcast?",
        "expected_source": "trustworthy_ai_podcast",
        "expected_keywords": ["trustworthy", "transparent", "accountable", "reliable"]
    },
    {
        "query": "How does the EU AI Act define an AI system?",
        "expected_source": "eu_ai_act",
        "expected_keywords": ["AI system", "machine-based", "infer", "content"]
    },
    {
        "query": "What is the role of human oversight in AI systems?",
        "expected_source": None,   # valid from either source
        "expected_keywords": ["human", "oversight", "control", "intervention"]
    },
    {
        "query": "What penalties exist for non-compliance with the EU AI Act?",
        "expected_source": "eu_ai_act",
        "expected_keywords": ["fine", "penalty", "EUR", "infringement"]
    }
]

print(f"Running evaluation on {len(EVAL_QUERIES)} benchmark queries...")

Running evaluation on 5 benchmark queries...


In [24]:
def evaluate_top1(
    query: str,
    expected_source: Optional[str],
    expected_keywords: List[str],
    reranker: str,
    n_retrieve: int = 10
) -> Dict:
    """
    Retrieve with the given reranker, check top-1 chunk for source match
    and keyword presence. Returns evaluation metrics.
    """
    hits = retrieve_baseline(query, n_results=n_retrieve)

    if reranker == "cross_encoder":
        hits = cross_encoder_rerank(query, hits)
    elif reranker == "llm":
        hits = llm_rerank(query, hits)

    top = hits[0]
    top_text   = top["text"].lower()
    top_source = top["metadata"]["source"]

    source_match   = (expected_source is None) or (top_source == expected_source)
    keyword_hits   = sum(1 for kw in expected_keywords if kw.lower() in top_text)
    keyword_recall = keyword_hits / max(len(expected_keywords), 1)

    return {
        "reranker":        reranker,
        "top_source":      top_source,
        "source_match":    source_match,
        "keyword_recall":  keyword_recall,
        "top_score":       top.get("ce_score_norm") or top.get("combined_score") or top["similarity_score"],
        "top_text_snippet": top["text"][:200]
    }


# Run evaluation across all queries and rerankers
eval_results = []
rerankers = ["none", "cross_encoder", "llm"]

for eq in EVAL_QUERIES:
    for rr in rerankers:
        metrics = evaluate_top1(
            query=eq["query"],
            expected_source=eq["expected_source"],
            expected_keywords=eq["expected_keywords"],
            reranker=rr
        )
        metrics["query"] = eq["query"][:60]
        eval_results.append(metrics)
        print(f"✓ [{rr:15}] Q: {eq['query'][:50]}... | source_match={metrics['source_match']} kw_recall={metrics['keyword_recall']:.2f}")

print("\n✅ Evaluation complete.")

✓ [none           ] Q: What are prohibited AI practices under the EU AI A... | source_match=True kw_recall=0.25
✓ [cross_encoder  ] Q: What are prohibited AI practices under the EU AI A... | source_match=True kw_recall=0.25
  🧠 LLM scoring 10 chunks...
✓ [llm            ] Q: What are prohibited AI practices under the EU AI A... | source_match=True kw_recall=0.50
✓ [none           ] Q: What does trustworthy AI mean according to the pod... | source_match=True kw_recall=0.25
✓ [cross_encoder  ] Q: What does trustworthy AI mean according to the pod... | source_match=True kw_recall=0.25
  🧠 LLM scoring 10 chunks...
✓ [llm            ] Q: What does trustworthy AI mean according to the pod... | source_match=True kw_recall=0.25
✓ [none           ] Q: How does the EU AI Act define an AI system?... | source_match=True kw_recall=0.25
✓ [cross_encoder  ] Q: How does the EU AI Act define an AI system?... | source_match=True kw_recall=0.25
  🧠 LLM scoring 10 chunks...
✓ [llm            ] Q: How does

In [25]:
# Aggregate performance summary
import pandas as pd

df = pd.DataFrame(eval_results)

summary = df.groupby("reranker").agg(
    source_accuracy  = ("source_match",   "mean"),
    avg_kw_recall    = ("keyword_recall",  "mean"),
).round(3)

print("\n" + "=" * 55)
print("       RETRIEVAL PERFORMANCE SUMMARY")
print("=" * 55)
print(summary.to_string())
print("=" * 55)
print("""
Metrics:
  source_accuracy : fraction of queries where top-1 chunk came
                    from the expected source document
  avg_kw_recall   : fraction of expected keywords found in top-1 chunk
""")


       RETRIEVAL PERFORMANCE SUMMARY
               source_accuracy  avg_kw_recall
reranker                                     
cross_encoder              1.0           0.35
llm                        1.0           0.40
none                       1.0           0.30

Metrics:
  source_accuracy : fraction of queries where top-1 chunk came
                    from the expected source document
  avg_kw_recall   : fraction of expected keywords found in top-1 chunk



In [26]:
# Per-query breakdown table
pivot = df.pivot_table(
    index="query",
    columns="reranker",
    values=["source_match", "keyword_recall"]
).round(2)

print("\nPer-query breakdown:")
print(pivot.to_string())


Per-query breakdown:
                                                            keyword_recall              source_match          
reranker                                                     cross_encoder   llm  none cross_encoder  llm none
query                                                                                                         
How does the EU AI Act define an AI system?                           0.25  0.50  0.25           1.0  1.0  1.0
What are prohibited AI practices under the EU AI Act?                 0.25  0.50  0.25           1.0  1.0  1.0
What does trustworthy AI mean according to the podcast?               0.25  0.25  0.25           1.0  1.0  1.0
What is the role of human oversight in AI systems?                    0.50  0.25  0.25           1.0  1.0  1.0
What penalties exist for non-compliance with the EU AI Act?           0.50  0.50  0.50           1.0  1.0  1.0


## Example Queries — Full RAG Answers

Demonstrating improved answer quality using cross-encoder reranking.

In [27]:
DEMO_QUERIES = [
    "What obligations do providers of high-risk AI systems have under the EU AI Act?",
    "How does the podcast describe the blueprint for trustworthy AI?",
    "What is the EU AI Act's approach to transparency for general-purpose AI models?"
]

for q in DEMO_QUERIES:
    print("\n" + "="*70)
    print(f"QUERY: {q}")
    print("="*70)

    r_none = rag_pipeline(q, n_retrieve=10, top_k=4, reranker="none")
    r_ce   = rag_pipeline(q, n_retrieve=10, top_k=4, reranker="cross_encoder")

    print("\n--- WITHOUT reranking ---")
    print(r_none["answer"])
    print("\n--- WITH cross-encoder reranking ---")
    print(r_ce["answer"])


QUERY: What obligations do providers of high-risk AI systems have under the EU AI Act?

--- WITHOUT reranking ---
Providers of high-risk AI systems have the following obligations under the EU AI Act:

1. Ensure compliance with the requirements set out in Section 2.
2. Indicate their name, registered trade name or trademark, and contact address on the high-risk AI system or, if not possible, on its packaging or accompanying documentation.
3. Have a quality management system in place.

(Source: eu_ai_act | Page: 62)

--- WITH cross-encoder reranking ---
Providers of high-risk AI systems have several obligations under the EU AI Act, including:

1. They must ensure that AI systems intended to interact directly with natural persons are designed and developed in a way that informs those persons that they are interacting with an AI system, unless this is obvious (Article 50).

2. They are considered providers and are subject to the obligations outlined in Article 16 if they put their name or

## Summary

| Approach | Pros | Cons |
|---|---|---|
| Embedding similarity only | Fast, no extra API calls | Can miss semantically relevant but lexically different chunks |
| LLM reranking | Flexible, handles nuance | Slow, expensive, token-intensive |
| Cross-encoder reranking | Accurate, fast inference, free | Requires local model, slower than embedding-only |

**Key findings:**
- Cross-encoder reranking consistently improves source accuracy and keyword recall for legal queries
- LLM scoring adds interpretability (reason field) but is costly at scale
- Metadata filtering (by source or section) further focuses retrieval and reduces noise
- Reranking helps most when embedding similarity retrieves partially relevant chunks that need re-ordering

**When to use reranking:**
- High-precision domains (legal, medical, compliance)
- Mixed-source corpora where source identity matters
- Queries requiring semantic nuance beyond keyword matching

See `lab_summary.md` for the written narrative.